# ARIA Phase 1 — Core Concepts
## Learning notebook: 4 things every AI engineer needs to understand

This notebook covers the 4 foundational concepts behind the Sponsor Brief Generator:

1. **System vs User Prompt** — how you give the model a role and constrain its behaviour
2. **Structured Output** — getting reliable JSON back instead of free-form text
3. **Token Tracking** — measuring what each call costs (critical for production)
4. **Prompt Chaining** — passing the output of one step as the input to the next

Each section has:
- A short explanation of the concept
- A runnable cell you can experiment with
- A small challenge to try yourself

---
> **Run cells top to bottom.** Each section builds on the last.

## Setup — load API key and create client

In [1]:
import sys
import os

# Add the project root to the path so we can import our own modules later
sys.path.insert(0, os.path.abspath(".."))

# Load .env file (same as what the FastAPI app does at startup)
from dotenv import load_dotenv
load_dotenv("../.env")

import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = "claude-sonnet-4-6"

print("Client ready. Model:", MODEL)

Client ready. Model: claude-sonnet-4-6


---
## Concept 1 — System vs User Prompt

**The mental model:**

| Role | Analogy | What it does |
|---|---|---|
| `system` | Your manager's standing instructions | Gives the model a persona, rules, and output format that apply to every message |
| `user` | The actual request in a conversation | The specific question or data for this particular call |

The model reads the `system` prompt first, then the `user` message. This means:
- Put **stable instructions** (persona, output format, constraints) in `system`
- Put **variable data** (company name, trial list) in `user`

**Why this matters for ARIA:** Every brief call uses the same analyst persona and the same JSON schema. That belongs in `system`. The sponsor data changes every call — that belongs in `user`.

Below: the same question asked *with* and *without* a system prompt, so you can see the difference.

In [2]:
# --- WITHOUT a system prompt ---
# The model has no context about who it is or what format to use.

response_no_system = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": "Tell me about Pfizer's clinical pipeline."
    }]
)

print("=== NO SYSTEM PROMPT ===")
print(response_no_system.content[0].text)

=== NO SYSTEM PROMPT ===
# Pfizer's Clinical Pipeline

Pfizer maintains one of the largest pharmaceutical pipelines in the industry. Here's an overview of key areas, though I should note my knowledge has a **cutoff of early 2025**, so current status may have changed.

---

## Key Therapeutic Areas

### Oncology
- One of Pfizer's largest investment areas following the **Seagen acquisition (2023)**
- Antibody-drug conjugates (ADCs) are a major focus
- Key candidates targeting various cancers including breast, bladder, and lung cancers
- TUKYSA (tucatinib) combinations for HER2+ cancers

### Vaccines
- mRNA vaccine technology development beyond COVID
- Influenza vaccine candidates
- RSV vaccines
- Lyme disease vaccine (VLA15/Vangueria) in late-stage trials

### Internal Medicine
- Obesity/metabolic disease treatments (competing in the GLP-1 space)
- Danuglipron (oral GLP-1 receptor agonist) - faced some development challenges

### Rare Diseases
- Gene therapy programs
- Hemophilia treatme

In [ ]:
# --- WITH a system prompt ---
# Same user question. Watch how the tone, focus, and format change.

response_with_system = client.messages.create(
    model=MODEL,
    max_tokens=300,
    system="""You are a senior BD analyst at a CRO. 
Your job is to identify outsourcing opportunities in a pharma company's pipeline.
Be concise. Focus only on what would require external CRO support.
Always end with one concrete recommended next action.""",
    messages=[{
        "role": "user",
        "content": "Tell me about Pfizer's clinical pipeline."
    }]
)

print("=== WITH SYSTEM PROMPT ===")
print(response_with_system.content[0].text)

=== WITH SYSTEM PROMPT ===
# Pfizer Clinical Pipeline – CRO Outsourcing Analysis

## Key Pipeline Areas (Current Focus)

**Oncology** – Largest segment; multiple Phase I/II/III trials across solid tumors and hematology (e.g., CDK inhibitors, ADCs, PARP inhibitors)

**Vaccines** – RSV, influenza, mRNA-based programs post-COVID

**Rare Disease** – Gene therapy programs (hemophilia, DMD) requiring specialized sites

**Internal Medicine** – Cardiovascular, metabolic disease candidates

---

## Likely Outsourcing Pressure Points

- **Gene therapy trials** – Highly specialized; limited internal infrastructure → **strong CRO opportunity**
- **Oncology Phase I** – Site network breadth needed; biomarker-heavy protocols
- **Geographic expansion** – Ex-US markets for global registrational trials
- **Post-COVID capacity normalization** – Pfizer has been actively **reducing headcount**, suggesting increased outsourcing appetite

---

## Caveats
My knowledge has a cutoff and Pfizer's pipeline update

**Challenge 1:** Edit the system prompt above — change the persona to a *competitor intelligence analyst* instead of a BD analyst. Re-run and observe how the focus of the response shifts. You're not changing the user message at all.

---
## Concept 2 — Structured Output

Free-form text is fine for a chatbot. For ARIA, the output feeds into a UI and downstream logic — so it must be machine-readable JSON every single time.

**The problem:** LLMs are trained to be helpful and conversational. By default they'll say things like *"Here is the brief you requested:"* before the JSON, or add markdown fences. Your JSON parser will break.

**The solution — two layers of defence:**

1. **Instruction in the system prompt:** `Return ONLY a valid JSON object. No markdown, no explanation.`
2. **Defensive stripping in Python:** even if the model adds fences, strip them before parsing.

Below: watch what happens without the instruction, then with it.

In [5]:
import json

# --- WITHOUT explicit JSON instruction ---
# The model will likely wrap the JSON in prose or markdown fences.

r = client.messages.create(
    model=MODEL,
    max_tokens=300,
    system="You are a CRO analyst.",
    messages=[{
        "role": "user",
        "content": 'Return a JSON object with keys "company" and "trial_count" for Pfizer.'
    }]
)

raw = r.content[0].text
print("Raw output:")
print(raw)
print()

# Try parsing — this will likely fail or require cleanup
try:
    parsed = json.loads(raw)
    print("Parsed OK:", parsed)
except json.JSONDecodeError as e:
    print("Parse FAILED:", e)

Raw output:
```json
{
  "company": "Pfizer",
  "trial_count": null
}
```

**Note:** I don't have access to a real-time database of clinical trials, so I cannot provide an accurate `trial_count`. To get the actual number of Pfizer clinical trials, I recommend checking:

- **ClinicalTrials.gov** – search by sponsor "Pfizer"
- **WHO ICTRP** – International Clinical Trials Registry Platform
- **Pfizer's own clinical trial portal**

Would you like me to help you structure a query to retrieve this data from one of those sources?

Parse FAILED: Expecting value: line 1 column 1 (char 0)


In [9]:
# --- WITH explicit JSON instruction + defensive stripping ---

def parse_llm_json(text: str) -> dict:
    """
    Robustly parse JSON from an LLM response.
    Handles: plain JSON, ```json fences, ``` fences, leading prose.
    """
    text = text.strip()

    # Strip markdown fences if present
    if text.startswith("```"):
        lines = text.split("\n")
        # Remove opening fence (```json or ```) and closing fence (```)
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()

    return json.loads(text)


r2 = client.messages.create(
    model=MODEL,
    max_tokens=300,
    system="""You are a CRO analyst.
CRITICAL: Return ONLY a valid JSON object. No markdown fences, no explanation, no extra text.""",
    messages=[{
        "role": "user",
        "content": 'Return a JSON object with keys "company", "trial_count" and "therapeutic_area" for Pfizer.'
    }]
)

raw2 = r2.content[0].text
print("Raw output:")
print(raw2)
print()

try:
    parsed2 = parse_llm_json(raw2)
    print("Parsed OK:", parsed2)
except json.JSONDecodeError as e:
    print("Parse FAILED:", e)

Raw output:
{"company": "Pfizer", "trial_count": 96, "therapeutic_area": "Oncology, Vaccines, Rare Disease, Inflammation & Immunology, Internal Medicine"}

Parsed OK: {'company': 'Pfizer', 'trial_count': 96, 'therapeutic_area': 'Oncology, Vaccines, Rare Disease, Inflammation & Immunology, Internal Medicine'}


**Challenge 2:** Add a third key `"therapeutic_areas"` (a list) to the schema in the user prompt. Update `parse_llm_json` to also validate that all three keys are present after parsing — raise a `ValueError` if any are missing. This is what the `SponsorBrief` Pydantic model in the real app does automatically.

---
## Concept 3 — Token Tracking

Every API call has a cost. In production you need to know:
- How much does one brief cost?
- Which part of the prompt consumes the most tokens?
- Are we within the model's context window?

The Anthropic SDK returns a `usage` object on every response with `input_tokens` and `output_tokens`.

**Pricing reference (as of 2026):**

| Model | Input | Output |
|---|---|---|
| claude-sonnet-4-6 | $3 / 1M tokens | $15 / 1M tokens |

The ARIA reference doc targets **< $0.15 per full run**. Let's measure whether we're on track.

In [10]:
# Pricing per million tokens (update if Anthropic changes rates)
PRICE_PER_M_INPUT  = 3.00   # USD
PRICE_PER_M_OUTPUT = 15.00  # USD

def calculate_cost(usage) -> dict:
    input_cost  = (usage.input_tokens  / 1_000_000) * PRICE_PER_M_INPUT
    output_cost = (usage.output_tokens / 1_000_000) * PRICE_PER_M_OUTPUT
    return {
        "input_tokens":  usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "total_tokens":  usage.input_tokens + usage.output_tokens,
        "input_cost_usd":  round(input_cost, 6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd":  round(input_cost + output_cost, 6),
    }

# Make a realistic brief-generation call and measure it
SAMPLE_SYSTEM = """You are a CRO BD analyst.
CRITICAL: Return ONLY a valid JSON object. No markdown fences, no explanation.
Schema: {"company_overview": "...", "opportunity_signals": [...], "recommended_service_areas": [...]}"""

SAMPLE_TRIALS = "\n".join([
    "- [NCT001] Phase 2 NSCLC immunotherapy | RECRUITING",
    "- [NCT002] Phase 3 breast cancer CDK4/6 inhibitor | ACTIVE_NOT_RECRUITING",
    "- [NCT003] Phase 1 rare disease gene therapy | RECRUITING",
    "- [NCT004] Phase 2 diabetes GLP-1 agonist | RECRUITING",
    "- [NCT005] Phase 3 Alzheimer's anti-amyloid | ACTIVE_NOT_RECRUITING",
])

r3 = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system=SAMPLE_SYSTEM,
    messages=[{
        "role": "user",
        "content": f"Company: Pfizer\n\nActive trials:\n{SAMPLE_TRIALS}\n\nGenerate the brief."
    }]
)

cost = calculate_cost(r3.usage)
print("Token usage:")
for k, v in cost.items():
    print(f"  {k}: {v}")

print(f"\nTarget per run: $0.1500")
print(f"This call cost: ${cost['total_cost_usd']:.4f}")
print(f"Budget remaining for other steps: ${0.15 - cost['total_cost_usd']:.4f}")

Token usage:
  input_tokens: 196
  output_tokens: 490
  total_tokens: 686
  input_cost_usd: 0.000588
  output_cost_usd: 0.00735
  total_cost_usd: 0.007938

Target per run: $0.1500
This call cost: $0.0079
Budget remaining for other steps: $0.1421


**Challenge 3:** In the real app, we pass up to 30 trials. Change `SAMPLE_TRIALS` above to include 20 fake trial entries (copy-paste and vary them). Re-run and observe how input tokens scale. At what trial count does this call alone exceed $0.05?

> **Key insight:** Input tokens (the prompt) usually dominate cost in RAG and agentic systems — not output. This is why the real `brief_generator.py` caps trials at 30 and only sends the first 30.

---
## Concept 4 — Prompt Chaining

A single LLM call can't do everything well. The pattern for ARIA is:

```
Step 1: fetch_trials()       → raw trial JSON from ClinicalTrials.gov
Step 2: fetch_sec_filings()  → raw SEC filing metadata from EDGAR
Step 3: generate_brief()     → pass both to Claude → structured brief
```

Each step's output becomes the next step's input. This is **prompt chaining** — also called a pipeline.

**Why not just ask Claude to "research Pfizer and give me a brief"?**
- Claude's knowledge has a cutoff date — it doesn't know about trials that started last month
- We control exactly what data goes in, so the output is traceable and auditable
- Each step can fail independently and be retried without redoing everything

Below: the full chain assembled manually, step by step.

In [11]:
import asyncio
import httpx

# ── Step 1: Fetch trials from ClinicalTrials.gov ──────────────────────────

async def fetch_trials(company_name: str, max_results: int = 10) -> list:
    params = {
        "query.spons": company_name,
        "filter.overallStatus": "RECRUITING,ACTIVE_NOT_RECRUITING",
        "fields": "NCTId,BriefTitle,Phase,Condition,OverallStatus",
        "pageSize": max_results,
        "format": "json",
    }
    async with httpx.AsyncClient(timeout=30.0) as client:
        r = await client.get("https://clinicaltrials.gov/api/v2/studies", params=params)
        r.raise_for_status()
        data = r.json()

    results = []
    for study in data.get("studies", []):
        proto = study.get("protocolSection", {})
        id_mod = proto.get("identificationModule", {})
        status_mod = proto.get("statusModule", {})
        design_mod = proto.get("designModule", {})
        conds_mod = proto.get("conditionsModule", {})
        phases = design_mod.get("phases", [])
        results.append({
            "nct_id":    id_mod.get("nctId"),
            "title":     id_mod.get("briefTitle"),
            "phase":     phases[0] if phases else None,
            "status":    status_mod.get("overallStatus"),
            "condition": (conds_mod.get("conditions") or [None])[0],
        })
    return results

# Run step 1 — edit the company name to try different sponsors
COMPANY = "Regeneron"
trials = await fetch_trials(COMPANY)

print(f"Step 1 complete: {len(trials)} trials fetched for {COMPANY}")
for t in trials[:5]:
    print(f"  [{t['nct_id']}] {t['phase']} | {t['condition']} | {t['status']}")

Step 1 complete: 10 trials fetched for Regeneron
  [NCT06834360] PHASE3 | Chronic Rhinosinusitis With Nasal Polyps | RECRUITING
  [NCT07316114] None | Chronic Spontaneous Urticaria | RECRUITING
  [NCT06452771] PHASE1 | Healthy Volunteers With Hyperlipidemia | ACTIVE_NOT_RECRUITING
  [NCT06421636] PHASE2 | Non-Transfusion Dependent Beta-Thalassemia (NTDT) | RECRUITING
  [NCT06887088] PHASE2 | Melanoma BRAF V600E/K Mutated | RECRUITING


In [12]:
# ── Step 2: Build a compact prompt context from step 1's output ──────────
# 
# This is the "clean" step in the chain. Raw API data is verbose — we extract
# only what the LLM needs to minimise tokens and reduce hallucination risk.

def build_trial_context(trials: list) -> str:
    if not trials:
        return "No active trials found."
    lines = []
    for t in trials:
        lines.append(
            f"- [{t['nct_id']}] Phase: {t['phase'] or 'N/A'} | "
            f"Condition: {t['condition'] or 'N/A'} | Status: {t['status']}"
        )
    return "\n".join(lines)

trial_context = build_trial_context(trials)
print("Step 2 complete: trial context ready for LLM")
print()
print(trial_context)

Step 2 complete: trial context ready for LLM

- [NCT06834360] Phase: PHASE3 | Condition: Chronic Rhinosinusitis With Nasal Polyps | Status: RECRUITING
- [NCT07316114] Phase: N/A | Condition: Chronic Spontaneous Urticaria | Status: RECRUITING
- [NCT06452771] Phase: PHASE1 | Condition: Healthy Volunteers With Hyperlipidemia | Status: ACTIVE_NOT_RECRUITING
- [NCT06421636] Phase: PHASE2 | Condition: Non-Transfusion Dependent Beta-Thalassemia (NTDT) | Status: RECRUITING
- [NCT06887088] Phase: PHASE2 | Condition: Melanoma BRAF V600E/K Mutated | Status: RECRUITING
- [NCT06444880] Phase: PHASE2 | Condition: SMARCB1-Deficient Malignancies | Status: RECRUITING
- [NCT04988074] Phase: PHASE2 | Condition: HPV-Related Squamous Cell Carcinoma | Status: RECRUITING
- [NCT05961709] Phase: PHASE2 | Condition: Colon Cancer | Status: RECRUITING
- [NCT06299111] Phase: PHASE2 | Condition: Venous Thromboembolism | Status: ACTIVE_NOT_RECRUITING
- [NCT06413680] Phase: PHASE1 | Condition: Melanoma | Status: RECR

In [13]:
# ── Step 3: Pass cleaned context to Claude, get structured brief ──────────
# This is where all 3 prior concepts come together:
#   - System prompt (Concept 1) sets the analyst persona
#   - JSON instruction (Concept 2) forces structured output
#   - We track tokens (Concept 3) on the result

CHAIN_SYSTEM = """You are a senior CRO BD analyst.
CRITICAL: Return ONLY a valid JSON object. No markdown fences, no explanation.
Schema:
{
  "company_overview": "2-3 sentence summary",
  "opportunity_signals": ["signal 1", "signal 2"],
  "recommended_service_areas": ["service 1", "service 2"]
}"""

r4 = client.messages.create(
    model=MODEL,
    max_tokens=600,
    system=CHAIN_SYSTEM,
    messages=[{
        "role": "user",
        "content": f"Company: {COMPANY}\n\nActive trials:\n{trial_context}\n\nGenerate the BD brief."
    }]
)

# Parse and display
brief = parse_llm_json(r4.content[0].text)
chain_cost = calculate_cost(r4.usage)

print(f"Step 3 complete: brief generated for {COMPANY}")
print(f"Cost: ${chain_cost['total_cost_usd']:.5f} ({chain_cost['total_tokens']} tokens)")
print()
print("=== COMPANY OVERVIEW ===")
print(brief["company_overview"])
print()
print("=== OPPORTUNITY SIGNALS ===")
for s in brief["opportunity_signals"]:
    print(f"  → {s}")
print()
print("=== RECOMMENDED SERVICE AREAS ===")
for s in brief["recommended_service_areas"]:
    print(f"  • {s}")

Step 3 complete: brief generated for Regeneron
Cost: $0.00930 (996 tokens)

=== COMPANY OVERVIEW ===
Regeneron Pharmaceuticals is a leading biotechnology company known for its robust pipeline spanning immunology, oncology, and cardiovascular/metabolic diseases, with major commercial products including Dupixent and Eylea. The company maintains an active clinical development program with trials ranging from Phase 1 through Phase 3 across multiple therapeutic areas. Their pipeline reflects a strong focus on both biologics and novel small molecule approaches, particularly in immuno-oncology and rare diseases.

=== OPPORTUNITY SIGNALS ===
  → Active Phase 3 recruiting trial in Chronic Rhinosinusitis With Nasal Polyps suggests Dupixent label expansion efforts requiring robust patient recruitment and endpoint management support
  → Multiple simultaneous Phase 2 oncology trials (melanoma BRAF-mutated, SMARCB1-deficient malignancies, HPV-related SCC, colon cancer) indicate a rapidly scaling imm

In [24]:
# ── Step 4: Pass cleaned context to Claude, get structured brief ──────────
# This is where all 3 prior concepts come together:
#   - System prompt (Concept 1) sets the analyst persona
#   - JSON instruction (Concept 2) forces structured output
#   - We track tokens (Concept 3) on the result

opportunity_signal_context = ""
for s in brief["opportunity_signals"]:
   opportunity_signal_context += f"  → {s}\n"


CHAIN_SYSTEM_4 = """You are a senior CRO BD analyst.
CRITICAL: Return ONLY a valid JSON object. No markdown fences, no explanation.
Schema:
{
  "company_name": "name of the company",
  "email_opening": "**one-paragraph cold outreach email opening** referencing the top signal."
}"""

r5 = client.messages.create(
    model=MODEL,
    max_tokens=600,
    system=CHAIN_SYSTEM_4,
    messages=[{
        "role": "user",
        "content": f"Company: {COMPANY}\n\nOpportunity Signals:\n{opportunity_signal_context}\n\nGenerate the outreach email opening."
    }]
)

# Parse and display
outreach_response = parse_llm_json(r5.content[0].text)
chain_cost = calculate_cost(r5.usage)

print(f"Step 4 complete: Outreach generated for {COMPANY}")
print(f"Cost: ${chain_cost['total_cost_usd']:.5f} ({chain_cost['total_tokens']} tokens)")
print()
print("=== Outreach Opening ===")
print(outreach_response["email_opening"])
print()

Step 4 complete: Outreach generated for Regeneron
Cost: $0.00445 (556 tokens)

=== Outreach Opening ===
Regeneron's pipeline momentum is unmistakable — from the Phase 3 Dupixent label expansion in Chronic Rhinosinusitis With Nasal Polyps to your rapidly scaling immuno-oncology portfolio spanning BRAF-mutated melanoma, SMARCB1-deficient malignancies, HPV-related SCC, and colon cancer, all running concurrently alongside your new rare disease push into Non-Transfusion Dependent Beta-Thalassemia and continued cardiovascular investment in hyperlipidemia and venous thromboembolism. Managing that breadth of simultaneous recruiting trials across phases, indications, and highly specialized patient populations is an extraordinary operational challenge — and it's precisely where our CRO's dedicated site networks, biomarker-driven patient identification capabilities, and integrated decentralized trial infrastructure have helped sponsors like yours compress enrollment timelines without sacrificing 

**Challenge 4 (capstone):** Add a 4th step to the chain.

After generating the brief, make a *second* LLM call that takes `brief["opportunity_signals"]` as input and generates a **one-paragraph cold outreach email opening** referencing the top signal. This is exactly what the `outreach_drafter` agent does in Phase 2 — you're building a simplified version of it here.

Requirements:
- New system prompt for a different persona: "BD rep writing a cold email"
- Pass only the signals (not the full brief) to keep tokens low
- Track cost for this second call separately, then print total cost for both calls combined

---

## Summary

| Concept | What you learned | Where it lives in the real app |
|---|---|---|
| System prompt | Shapes persona, format, constraints — separate from data | `SYSTEM_PROMPT` in `brief_generator.py` |
| Structured output | JSON instruction + defensive stripping | `parse_llm_json` / `json.loads` in `brief_generator.py` |
| Token tracking | `usage.input_tokens` + `usage.output_tokens` × price | `UsageLog` returned in every `BriefResponse` |
| Prompt chaining | Fetch → clean → generate — each step feeds the next | `routers/brief.py` calling 3 services in sequence |

You're now ready to look at the production code and understand every decision. Open [`backend/services/brief_generator.py`](../backend/services/brief_generator.py) — it's the same 3-step chain you just built, with Pydantic validation added on top.

---
## Concept 5 — Tool Calling

So far Claude has only received text and returned text. **Tool calling** lets you give Claude a set of Python functions it can choose to invoke — the model decides *when* to call them and *what arguments* to pass.

**The pattern (3 steps):**
1. You define tools as JSON schemas and pass them to `client.messages.create(tools=[...])`
2. Claude returns a `tool_use` block instead of text when it wants to call a function
3. You run the actual function and send the result back as a `tool_result` message
4. Claude uses the result to generate its final answer

**Why this matters for ARIA:**
- Phase 1 calls ClinicalTrials.gov directly in Python — the model never touches it
- In Phase 2, the Intel Agent uses web search *as a tool* — Claude decides what to search for and when
- Tool calling is the bridge between an LLM that generates text and an agent that takes actions

Below: give Claude a `get_trial_count` tool and watch it decide to use it.

In [34]:
import httpx, asyncio, json

# ── Step 1: Define the tool schema ────────────────────────────────────────
# This is what you give to the Anthropic API.
# It's a JSON schema — like a Pydantic model but in dict form.
# Claude reads this to know what the function does and what arguments to pass.

tools = [
    {
        "name": "get_trial_count",
        "description": (
            "Fetches the number of active clinical trials for a pharma sponsor "
            "from ClinicalTrials.gov. Use this when you need current trial data."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "The pharma/biotech company name to search for",
                },
                "phase": {
                    "type": "string",
                    "description": "Optional trial phase filter, e.g. 'PHASE2'",
                },
            },
            "required": ["company_name"],
        },
    }
]

# ── Step 2: The actual Python function the tool maps to ───────────────────
# Claude never runs this directly — YOU run it when Claude asks for it.

async def get_trial_count(company_name: str, phase: str = None) -> dict:
    params = {
        "query.spons": company_name,
        "filter.overallStatus": "RECRUITING,ACTIVE_NOT_RECRUITING",
        "pageSize": 1,         # we only need the total count, not the records
        "format": "json",
        "countTotal": "true",
    }
    # if phase:
    #     params["filter.phase"] = phase.upper()

    async with httpx.AsyncClient(timeout=15.0) as http:
        r = await http.get("https://clinicaltrials.gov/api/v2/studies", params=params)
        r.raise_for_status()
        data = r.json()

    total = data.get("totalCount", 0)
    return {"company": company_name,  "active_trial_count": total}


# ── Step 3: First API call — Claude sees the question + available tools ───
response1 = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=tools,
    messages=[{
        "role": "user",
        "content": "How many active Phase 2 trials does Regeneron currently have?",
    }]
)

print("Stop reason:", response1.stop_reason)
print("Content blocks:")
for block in response1.content:
    print(f"  type={block.type}", end="")
    if block.type == "tool_use":
        print(f"  name={block.name}  inputs={block.input}")
    else:
        print(f"  text={block.text[:80] if hasattr(block, 'text') else ''}")

Stop reason: tool_use
Content blocks:
  type=tool_use  name=get_trial_count  inputs={'company_name': 'Regeneron', 'phase': 'PHASE2'}


In [35]:
# ── Step 4: Run the tool Claude asked for, then send result back ──────────
# 
# When stop_reason == "tool_use", Claude is waiting. We must:
#   1. Extract the tool_use block to get the function name + arguments
#   2. Run our actual Python function with those arguments
#   3. Send the result back in a new "user" message as a tool_result block
#   4. Claude then uses the real data to answer

# Find the tool_use block in the response
tool_use_block = next(b for b in response1.content if b.type == "tool_use")

# print(tool_use_block)

# Run the actual function with the arguments Claude chose
tool_result = await get_trial_count(**tool_use_block.input)
print("Tool result from ClinicalTrials.gov:", tool_result)

# ── Step 5: Second API call — send the tool result back to Claude ─────────
# The messages list now contains the full conversation:
#   user question → Claude's tool_use request → our tool result → final answer

response2 = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=tools,
    messages=[
        # Original user question
        {"role": "user", "content": "How many active Phase 2 trials does Regeneron currently have?"},
        # Claude's response (including the tool_use block) — must be included as-is
        {"role": "assistant", "content": response1.content},
        # Our tool result
        {
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,      # must match the tool_use block's id
                "content": json.dumps(tool_result),
            }],
        },
    ],
)

print("\nClaude's final answer:")
print(response2.content[0].text)
print(f"\nTotal tokens used: {response1.usage.input_tokens + response1.usage.output_tokens + response2.usage.input_tokens + response2.usage.output_tokens}")

Tool result from ClinicalTrials.gov: {'company': 'Regeneron', 'active_trial_count': 202}

Claude's final answer:
Based on the latest data from ClinicalTrials.gov, **Regeneron** currently has **202 active Phase 2 clinical trials**. This reflects the company's broad and active research pipeline across multiple therapeutic areas. Would you like to know more details, such as comparisons with other companies or trials in different phases?

Total tokens used: 1559


In [39]:
# Challenge 5:** 
# Add a second tool called `get_sec_filing_count` that hits the EDGAR API (already implemented in `backend/services/sec_edgar.py`). 
# Give Claude both tools and ask: *"Is Novo Nordisk a publicly traded company, and how many oncology trials do they have active?"* 
# — Claude should call both tools before answering.

# ── Step 1: Define the tool schema ────────────────────────────────────────
# This is what you give to the Anthropic API.
# It's a JSON schema — like a Pydantic model but in dict form.
# Claude reads this to know what the function does and what arguments to pass.

tools = [
    {
        "name": "get_sec_filing_count",
        "description": (
            "Search SEC EDGAR for recent 10-K/10-Q filings by the company."
            "Returns filing metadata (dates, form types) as a signal of SEC presence."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "The pharma/biotech company name to search for",
                }
            },
            "required": ["company_name"],
        },
    }
]





# ── Step 2: The actual Python function the tool maps to ───────────────────
# Claude never runs this directly — YOU run it when Claude asks for it.

async def get_sec_filing_count(company_name: str) -> dict:
    params = {
        "q": f'"{company_name}"',
        "dateRange": "custom",
        "startdt": "2023-01-01",
        "forms": "10-K,10-Q",
    }

    # EDGAR requires a User-Agent header identifying the caller
    EDGAR_HEADERS = {
        "User-Agent": "ARIA Research Tool aria-research@example.com",
        "Accept": "application/json",
    }

    async with httpx.AsyncClient(timeout=15.0) as http:
        r = await http.get("https://efts.sec.gov/LATEST/search-index", params=params,headers=EDGAR_HEADERS)
        r.raise_for_status()
        data = r.json()

    total = data.get("totalCount", 0)
    return {"company": company_name,  "active_trial_count": total}


# ── Step 3: First API call — Claude sees the question + available tools ───
response1 = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=tools,
    messages=[{
        "role": "user",
        "content": "Is Novo Nordisk a publicly traded company, and how many oncology trials do they have active?",
    }]
)

print(response1)


# ── Step 4: Run the tool Claude asked for, then send result back ──────────
# 
# When stop_reason == "tool_use", Claude is waiting. We must:
#   1. Extract the tool_use block to get the function name + arguments
#   2. Run our actual Python function with those arguments
#   3. Send the result back in a new "user" message as a tool_result block
#   4. Claude then uses the real data to answer

# Find the tool_use block in the response
tool_use_block = next(b for b in response1.content if b.type == "tool_use")

print(tool_use_block)

response2 = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=tools,
    messages=[
        # Original user question
        {"role": "user", "content": "Is Novo Nordisk a publicly traded company, and how many oncology trials do they have active?"},
        # Claude's response (including the tool_use block) — must be included as-is
        {"role": "assistant", "content": response1.content},
        # Our tool result
        {
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,      # must match the tool_use block's id
                "content": json.dumps(tool_result),
            }],
        },
    ],
)

print("\nClaude's final answer:")
print(response2.content[0].text)
print(f"\nTotal tokens used: {response1.usage.input_tokens + response1.usage.output_tokens + response2.usage.input_tokens + response2.usage.output_tokens}")

# print("Stop reason:", response1.stop_reason)
# print("Content blocks:")
# for block in response1.content:
#     print(f"  type={block.type}", end="")
#     if block.type == "tool_use":
#         print(f"  name={block.name}  inputs={block.input}")
#     else:
#         print(f"  text={block.text[:80] if hasattr(block, 'text') else ''}")



Message(id='msg_01Y5dKvhv3xAgrQgu78tJD99', container=None, content=[TextBlock(citations=None, text="I can help answer your question! Let me look up Novo Nordisk's SEC filing data to determine if they are publicly traded. However, please note that **I don't have a tool to search for active clinical trials**, so I'll only be able to retrieve SEC filing information for the public trading part of your question.\n\nLet me run that search now!", type='text'), ToolUseBlock(id='toolu_01HkZJ6xvr72UE2ShTL23Mej', caller=DirectCaller(type='direct'), input={'company_name': 'Novo Nordisk'}, name='get_sec_filing_count', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=633, output_tokens=139, server_tool_use=None, ser

**What just happened — the tool calling loop:**

```
You → Claude:  "How many Phase 2 trials does Regeneron have?" + tool definition
Claude → You:  stop_reason="tool_use", inputs={"company_name": "Regeneron", "phase": "PHASE2"}
You → API:     get_trial_count("Regeneron", "PHASE2")  ← real HTTP call
API → You:     {"active_trial_count": 14}
You → Claude:  tool_result with the count
Claude → You:  "Regeneron currently has 14 active Phase 2 clinical trials..."
```

Notice: **Claude decided** to filter by PHASE2 — you never told it to pass that argument explicitly. It inferred from the question that the phase filter was needed.

**Challenge 5:** Add a second tool called `get_sec_filing_count` that hits the EDGAR API (already implemented in `backend/services/sec_edgar.py`). Give Claude both tools and ask: *"Is Novo Nordisk a publicly traded company, and how many oncology trials do they have active?"* — Claude should call both tools before answering.

---

## Updated Summary

| Concept | What you built | Where in production |
|---|---|---|
| System prompt | Persona + output constraints | `SYSTEM_PROMPT` in `brief_generator.py` |
| Structured output | JSON instruction + `_strip_fences` + retry | `_parse_with_retry` in `brief_generator.py` |
| Token tracking | Cost per call including retry overhead | `UsageLog` in `BriefResponse` |
| Prompt chaining | Fetch → filter → generate | `routers/brief.py` → services |
| **Tool calling** | Schema definition → `tool_use` detection → result injection | Phase 2 Intel Agent with web search MCP |